In [2]:
import numpy as np
from scipy import integrate

def compute_lambda_coeffs(var1, var2, var3, var4, var5, var6, var7, var8, var9, var10):
    """
    Tính Lambda0, Lambda1, Lambda2 cho 1 bộ 10 biến đầu vào.
    var1..var10 chính là các biến bạn đang dùng trong code gốc.
    """

    # ============================================================================
    # Input Parameters (giữ nguyên ý nghĩa như code gốc của bạn)
    # ============================================================================
    a_over_h   = var1   # Ratio a/h
    b_over_a   = var2   # Ratio b/a
    T          = var3   # Temperature (K)
    gori_deg   = var4   # Degree of Gori folding 
    gori_wt    = var5   # GORi weight fraction
    DT         = var6   # Distribution
    Kun        = var7   # Upper spring stiffness
    Ksn        = var8   # Lower spring stiffness
    Kln        = var9   # Shear spring stiffness  
    CL         = var10  # Case of loading

    # Case of loading – mình giữ như bạn đang dùng (uniaxial y-edge, encode=1)
    if CL == 1:      # Uniaxial x-direction
        Chi1 = -1
        Chi2 = 0
    elif CL == 2:    # Biaxial
        Chi1 = -1
        Chi2 = -1
    else:             # Uniaxial y-direction (will not use)
        Chi1 = -1
        Chi2 = -1

    # ============================================================================
    # Material and geometrical parameters
    # ============================================================================
    h = 0.005  # Thickness default (giống code gốc)
    a = a_over_h * h
    b = b_over_a * a
    HGr = gori_deg / 100.0
    WGr = gori_wt / 100.0
    
    # Springs
    Kun = Kun
    Ksn = Ksn
    Kln = Kln

    # Mode numbers
    m = 1
    n = 1

    # Constants for HSDT formulation
    c1 = 4.0 / (h**2) / 3.0
    c2 = 3.0 * c1

    # ============================================================================
    # Material Properties - FG-GOEAM
    # ============================================================================
    T0 = 300.0
    DeltaT = T - T0

    # Material constants
    EGr = 929.57e9
    NuGr = 0.22
    AlphaGr = -3.98e-6
    RoGr = 1800.0
    lGr = 83.76e-10
    tGr = 3.4e-10

    ECu = 65.79e9
    NuCu = 0.387
    AlphaCu = 16.51e-6
    RoCu = 8800.0

    Dm = ECu * h**3 / 12.0 / (1.0 - NuCu**2)

    NL = 10  # number of layers

    # Graphene volume fraction distribution (U-W_Gr)
    VGrs = WGr / (WGr + (RoGr / RoCu) * (1.0 - WGr))

    # Initialize stiffness matrices
    A11=A22=A12=A21=A66=0.0
    B11=B22=B12=B21=B66=0.0
    D11=D22=D12=D21=D66=0.0
    E11=E22=E12=E21=E66=0.0
    F11=F22=F12=F21=F66=0.0
    H11=H22=H12=H21=H66=0.0
    A44=A55=D44=D55=F44=F55=0.0

    NxxT=NyyT=MxxT=MyyT=PxxT=PyyT=0.0

    # ----------------- Loop through layers (giống code gốc) --------------------
    for k in range(1, NL + 1):
        # ------------- chọn distribution theo DT -------------
        if DT == 1:
            # U-W_Gr (Uniform)
            VGr = VGrs
        elif DT == 2:
            # X-W_Gr pattern
            # VGr = 2 * VGrs * abs(2*k - 1 - NL) / NL
            VGr = 2.0 * VGrs * abs(2 * k - 1 - NL) / NL
        elif DT == 3:
            # O-W_Gr pattern
            # VGr = 2 * VGrs * (1 - (1/NL) * abs(-1 + 2*k - NL))
            VGr = 2.0 * VGrs * (1.0 - (1.0 / NL) * abs(-1 + 2 * k - NL))
        else :
            # A-X_Gr pattern
            # VGr = VGrs * abs(2k-1)/NL
            VGr = VGrs * abs(2 * k - 1) / NL
      

        VCu = 1.0 - VGr

        fE = (1.11 - 1.22*VGr - 0.134*(T/T0) + 0.559*VGr*(T/T0) -
              5.5*HGr*VGr + 38*HGr*VGr**2 - 20.6*HGr**2*VGr**2)
        fv = (1.01 - 1.43*VGr + 0.165*(T/T0) - 16.8*HGr*VGr -
              1.1*HGr*VGr*(T/T0) + 16*HGr**2*VGr**2)
        fAlpha = (0.794 - 16.8*VGr**2 - 0.0279*(T/T0)**2 +
                  0.182*(T/T0)*(1+VGr))

        Xi = 2.0 * lGr / tGr
        Eta = ((EGr/ECu) - 1.0) / ((EGr/ECu) + Xi)

        E = (1.0 + Xi*Eta*VGr) * ECu * fE / (1.0 - Eta*VGr)
        Nu = (NuGr*VGr + NuCu*VCu) * fv
        Alpha = (AlphaGr*VGr + AlphaCu*VCu) * fAlpha

        def Q11a(z): return E / (1 - Nu**2)
        def Q22a(z): return E / (1 - Nu**2)
        def Q12a(z): return E * Nu / (1 - Nu**2)
        def Q21a(z): return E * Nu / (1 - Nu**2)
        def Q66a(z): return E / (2 * (1 + Nu))

        def Q11b(z): return Q11a(z) * z
        def Q22b(z): return Q22a(z) * z
        def Q12b(z): return Q12a(z) * z
        def Q21b(z): return Q21a(z) * z
        def Q66b(z): return Q66a(z) * z

        def Q11d(z): return Q11a(z) * z**2
        def Q22d(z): return Q22a(z) * z**2
        def Q12d(z): return Q12a(z) * z**2
        def Q21d(z): return Q21a(z) * z**2
        def Q66d(z): return Q66a(z) * z**2

        def Q11e(z): return Q11a(z) * z**3
        def Q22e(z): return Q22a(z) * z**3
        def Q12e(z): return Q12a(z) * z**3
        def Q21e(z): return Q21a(z) * z**3
        def Q66e(z): return Q66a(z) * z**3

        def Q11f(z): return Q11a(z) * z**4
        def Q22f(z): return Q22a(z) * z**4
        def Q12f(z): return Q12a(z) * z**4
        def Q21f(z): return Q21a(z) * z**4
        def Q66f(z): return Q66a(z) * z**4

        def Q11h(z): return Q11a(z) * z**6
        def Q22h(z): return Q22a(z) * z**6
        def Q12h(z): return Q12a(z) * z**6
        def Q21h(z): return Q21a(z) * z**6
        def Q66h(z): return Q66a(z) * z**6

        def Q44a(z): return E / (2 * (1 + Nu))
        def Q55a(z): return E / (2 * (1 + Nu))
        def Q44d(z): return Q44a(z) * z**2
        def Q55d(z): return Q55a(z) * z**2
        def Q44f(z): return Q44a(z) * z**4
        def Q55f(z): return Q55a(z) * z**4

        def nx(z): return (Q11a(z) + Q12a(z)) * Alpha * DeltaT
        def ny(z): return (Q21a(z) + Q22a(z)) * Alpha * DeltaT
        def mx(z): return nx(z) * z
        def my(z): return ny(z) * z
        def px(z): return nx(z) * z**3
        def py(z): return ny(z) * z**3

        z_low  = (NL/2.0 - k)     * h / NL
        z_high = (NL/2.0 - k + 1) * h / NL

        nxt, _ = integrate.quad(nx, z_low, z_high)
        nyt, _ = integrate.quad(ny, z_low, z_high)
        mxt, _ = integrate.quad(mx, z_low, z_high)
        myt, _ = integrate.quad(my, z_low, z_high)
        pxt, _ = integrate.quad(px, z_low, z_high)
        pyt, _ = integrate.quad(py, z_low, z_high)

        a11, _ = integrate.quad(Q11a, z_low, z_high)
        a12, _ = integrate.quad(Q12a, z_low, z_high)
        a21, _ = integrate.quad(Q21a, z_low, z_high)
        a22, _ = integrate.quad(Q22a, z_low, z_high)
        a66, _ = integrate.quad(Q66a, z_low, z_high)

        b11, _ = integrate.quad(Q11b, z_low, z_high)
        b12, _ = integrate.quad(Q12b, z_low, z_high)
        b21, _ = integrate.quad(Q21b, z_low, z_high)
        b22, _ = integrate.quad(Q22b, z_low, z_high)
        b66, _ = integrate.quad(Q66b, z_low, z_high)

        d11, _ = integrate.quad(Q11d, z_low, z_high)
        d12, _ = integrate.quad(Q12d, z_low, z_high)
        d21, _ = integrate.quad(Q21d, z_low, z_high)
        d22, _ = integrate.quad(Q22d, z_low, z_high)
        d66, _ = integrate.quad(Q66d, z_low, z_high)

        e11, _ = integrate.quad(Q11e, z_low, z_high)
        e12, _ = integrate.quad(Q12e, z_low, z_high)
        e21, _ = integrate.quad(Q21e, z_low, z_high)
        e22, _ = integrate.quad(Q22e, z_low, z_high)
        e66, _ = integrate.quad(Q66e, z_low, z_high)

        f11, _ = integrate.quad(Q11f, z_low, z_high)
        f12, _ = integrate.quad(Q12f, z_low, z_high)
        f21, _ = integrate.quad(Q21f, z_low, z_high)
        f22, _ = integrate.quad(Q22f, z_low, z_high)
        f66, _ = integrate.quad(Q66f, z_low, z_high)

        h11, _ = integrate.quad(Q11h, z_low, z_high)
        h12, _ = integrate.quad(Q12h, z_low, z_high)
        h21, _ = integrate.quad(Q21h, z_low, z_high)
        h22, _ = integrate.quad(Q22h, z_low, z_high)
        h66, _ = integrate.quad(Q66h, z_low, z_high)

        a44, _ = integrate.quad(Q44a, z_low, z_high)
        a55, _ = integrate.quad(Q55a, z_low, z_high)
        d44, _ = integrate.quad(Q44d, z_low, z_high)
        d55, _ = integrate.quad(Q55d, z_low, z_high)
        f44, _ = integrate.quad(Q44f, z_low, z_high)
        f55, _ = integrate.quad(Q55f, z_low, z_high)

        A11 += a11; A22 += a22; A12 += a12; A21 += a21; A66 += a66
        B11 += b11; B22 += b22; B12 += b12; B21 += b21; B66 += b66
        D11 += d11; D22 += d22; D12 += d12; D21 += d21; D66 += d66
        E11 += e11; E22 += e22; E12 += e12; E21 += e21; E66 += e66
        F11 += f11; F22 += f22; F12 += f12; F21 += f21; F66 += f66
        H11 += h11; H22 += h22; H12 += h12; H21 += h21; H66 += h66
        A44 += a44; A55 += a55
        D44 += d44; D55 += d55
        F44 += f44; F55 += f55

        NxxT += nxt; NyyT += nyt
        MxxT += mxt; MyyT += myt
        PxxT += pxt; PyyT += pyt

    # ============================================================================
    # Kerr Foundation Parameters
    # ============================================================================
    Ku = Kun * Dm / (a**4)
    Kl = Kln * Dm / (a**4)
    Ks = Ksn * Dm / (a**2)

    K1 = Kl * Ku / (Kl + Ku)
    K2 = Ks * Ku / (Kl + Ku)

    ##################### Analytical =============================
    ########## %% Eq. (19)
    C11 = A12 / (A12 * A21 - A11 * A22)
    C12 = -(A22 / (A12 * A21 - A11 * A22))
    C13 = -((-A22 * B11 + A12 * B21) / (A12 * A21 - A11 * A22))
    C14 = -((-A22 * B12 + A12 * B22) / (A12 * A21 - A11 * A22))
    C15 = -((-A22 * E11 + A12 * E21) / (A12 * A21 - A11 * A22))
    C16 = -((-A22 * E12 + A12 * E22) / (A12 * A21 - A11 * A22))
    C21 = A11 / (-A12 * A21 + A11 * A22)
    C22 = -(A21 / (-A12 * A21 + A11 * A22))
    C23 = -((-A21 * B11 + A11 * B21) / (-A12 * A21 + A11 * A22))
    C24 = -((-A21 * B12 + A11 * B22) / (-A12 * A21 + A11 * A22))
    C25 = -((-A21 * E11 + A11 * E21) / (-A12 * A21 + A11 * A22))
    C26 = -((-A21 * E12 + A11 * E22) / (-A12 * A21 + A11 * A22))
    C31 = -(1 / A66)
    C32 = -(B66 / A66)
    C33 = -(E66 / A66)

    ############ %% Eq. (21)
    J11 = C21
    J12 = C11 + C22 - C31
    J13 = C12
    J21 = C23 - c1 * C25
    J22 = C13 - c1 * C15 - C32 + c1 * C33
    J23 = C24 - c1 * C26 - C32 + c1 * C33
    J24 = C14 - c1 * C16
    J31 = -c1 * C25
    J32 = -c1 * C15 - c1 * C26 + 2 * c1 * C33
    J33 = -c1 * C16

    ####### %% Appendix A
    S11 = -c1**2 * C15 * E11 - c1**2 * C25 * E12 - c1**2 * H11
    S12 = (-c1**2 * C16 * E11 - c1**2 * C26 * E12 - c1**2 * C15 * E21 - c1**2 * C25 * E22
          - 4 * c1**2 * C33 * E66 - c1**2 * H12 - c1**2 * H21 - 4 * c1**2 * H66)
    S13 = -c1**2 * C16 * E21 - c1**2 * C26 * E22 - c1**2 * H22
    S14 = A55 - 2 * c2 * D55 + c2**2 * F55
    S15 = A44 - 2 * c2 * D44 + c2**2 * F44
    S16 = A55 - 2 * c2 * D55 + c2**2 * F55
    S17 = (c1 * C13 * E11 - c1**2 * C15 * E11 + c1 * C23 * E12 - c1**2 * C25 * E12
          + c1 * F11 - c1**2 * H11)
    S18 = (c1 * C13 * E21 - c1**2 * C15 * E21 + c1 * C23 * E22 - c1**2 * C25 * E22
          + 2 * c1 * C32 * E66 - 2 * c1**2 * C33 * E66 + c1 * F21 + 2 * c1 * F66
          - c1**2 * H21 - 2 * c1**2 * H66)
    S19 = A44 - 2 * c2 * D44 + c2**2 * F44
    S110 = (c1 * C14 * E21 - c1**2 * C16 * E21 + c1 * C24 * E22 - c1**2 * C26 * E22
            + c1 * F22 - c1**2 * H22)
    S111 = (c1 * C14 * E11 - c1**2 * C16 * E11 + c1 * C24 * E12 - c1**2 * C26 * E12
            + 2 * c1 * C32 * E66 - 2 * c1**2 * C33 * E66 + c1 * F12 + 2 * c1 * F66
            - c1**2 * H12 - 2 * c1**2 * H66)
    S112 = c1 * C11 * E11 + c1 * C21 * E12
    S113 = c1 * C12 * E11 + c1 * C22 * E12 + c1 * C11 * E21 + c1 * C21 * E22 + 2 * c1 * C31 * E66
    S114 = c1 * C12 * E21 + c1 * C22 * E22

    S21 = (-B11 * c1 * C16 - B12 * c1 * C26 - 2 * B66 * c1 * C33 + c1**2 * C16 * E11 +
          c1**2 * C26 * E12 + 2 * c1**2 * C33 * E66 - c1 * F12 - 2 * c1 * F66 + c1**2 * H12 +
          2 * c1**2 * H66)
    S22 = (-B11 * c1 * C15 - B12 * c1 * C25 + c1**2 * C15 * E11 + c1**2 * C25 * E12 -
          c1 * F11 + c1**2 * H11)
    S23 = -A55 + 2 * c2 * D55 - c2**2 * F55
    S24 = (B11 * C13 - B11 * c1 * C15 + B12 * C23 - B12 * c1 * C25 + D11 - c1 * C13 * E11 +
          c1**2 * C15 * E11 - c1 * C23 * E12 + c1**2 * C25 * E12 - 2 * c1 * F11 + c1**2 * H11)
    S25 = (B66 * C32 - B66 * c1 * C33 + D66 - c1 * C32 * E66 + c1**2 * C33 * E66 -
          2 * c1 * F66 + c1**2 * H66)
    S26 = -A55 + 2 * c2 * D55 - c2**2 * F55
    S27 = (B11 * C14 - B11 * c1 * C16 + B12 * C24 - B12 * c1 * C26 + B66 * C32 -
          B66 * c1 * C33 + D12 + D66 - c1 * C14 * E11 + c1**2 * C16 * E11 - c1 * C24 * E12 +
          c1**2 * C26 * E12 - c1 * C32 * E66 + c1**2 * C33 * E66 - 2 * c1 * F12 - 2 * c1 * F66 +
          c1**2 * H12 + c1**2 * H66)
    S28 = B11 * C11 + B12 * C21 - c1 * C11 * E11 - c1 * C21 * E12
    S29 = (B11 * C12 + B12 * C22 + B66 * C31 - c1 * C12 * E11 - c1 * C22 * E12 -
          c1 * C31 * E66)

    S31 = (-B21 * c1 * C15 - B22 * c1 * C25 - 2 * B66 * c1 * C33 + c1**2 * C15 * E21 +
          c1**2 * C25 * E22 + 2 * c1**2 * C33 * E66 - c1 * F21 - 2 * c1 * F66 + c1**2 * H21 +
          2 * c1**2 * H66)
    S32 = (-B21 * c1 * C16 - B22 * c1 * C26 + c1**2 * C16 * E21 + c1**2 * C26 * E22 -
          c1 * F22 + c1**2 * H22)
    S33 = -A44 + 2 * c2 * D44 - c2**2 * F44
    S34 = (B21 * C13 - B21 * c1 * C15 + B22 * C23 - B22 * c1 * C25 + B66 * C32 -
          B66 * c1 * C33 + D21 + D66 - c1 * C13 * E21 + c1**2 * C15 * E21 - c1 * C23 * E22 +
          c1**2 * C25 * E22 - c1 * C32 * E66 + c1**2 * C33 * E66 - 2 * c1 * F21 - 2 * c1 * F66 +
          c1**2 * H21 + c1**2 * H66)
    S35 = (B66 * C32 - B66 * c1 * C33 + D66 - c1 * C32 * E66 + c1**2 * C33 * E66 -
          2 * c1 * F66 + c1**2 * H66)
    S36 = (B21 * C14 - B21 * c1 * C16 + B22 * C24 - B22 * c1 * C26 + D22 - c1 * C14 * E21 +
          c1**2 * C16 * E21 - c1 * C24 * E22 + c1**2 * C26 * E22 - 2 * c1 * F22 + c1**2 * H22)
    S37 = -A44 + 2 * c2 * D44 - c2**2 * F44
    S38 = B21 * C12 + B22 * C22 - c1 * C12 * E21 - c1 * C22 * E22
    S39 = (B21 * C11 + B22 * C21 + B66 * C31 - c1 * C11 * E21 - c1 * C21 * E22 -
          c1 * C31 * E66)

    ######### %% Eq. (28)
    R11 = (a**2 * n**2) / (32 * b**2 * J11 * m**2)
    R21 = (b**2 * m**2) / (32 * a**2 * J13 * n**2)
    R31 = (-a * b**4 * J21 * m**3 - a**3 * b**2 * J22 * m * n**2) / \
          ((b**4 * J11 * m**4 + a**2 * b**2 * J12 * m**2 * n**2 + a**4 * J13 * n**4) * np.pi)
    R32 = (-a**2 * b**3 * J23 * m**2 * n - a**4 * b * J24 * n**3) / \
          ((b**4 * J11 * m**4 + a**2 * b**2 * J12 * m**2 * n**2 + a**4 * J13 * n**4) * np.pi)
    R33 = (-b**4 * J31 * m**4 * np.pi - a**2 * b**2 * J32 * m**2 * n**2 * np.pi - a**4 * J33 * n**4 * np.pi) / \
          ((b**4 * J11 * m**4 + a**2 * b**2 * J12 * m**2 * n**2 + a**4 * J13 * n**4) * np.pi)

    ########### %% Eq. (32)
    S2 = (1/8) * a**(-1) * b**(-1) * m**(-1) * np.pi**(-1) * (b**2 * K2 * m**2 * np.pi**2 +
          a**2 * (b**2 * K1 + K2 * n**2 * np.pi**2)) * ((-2) * m * np.pi + np.sin(2 * m * np.pi))

    # %% Eq. (31)
    V11 = (1 / (8 * a**3 * b**3 * m) * np.pi * (a**4 * n**4 * np.pi**2 * (R33 * S114 + S13) +
          b**4 * (m**4 * np.pi**2 * (S11 + R33 * S112) - a**2 * m**2 * S14) +
          a**2 * b**2 * n**2 * (m**2 * np.pi**2 * (R33 * S113 + S12) - a**2 * S15)) * (2 * m * np.pi - np.sin(2 * m * np.pi)))

    V12 = (-(1 / (3 * a**3 * b**3 * m * n)) * 4 * np.pi**2 * (-((1 + 2 * (-1)**n) * a**2 * n**2 * (-b**2 * m**2 * R33 +
          16 * a**2 * n**2 * R21 * S114)) + b**2 * m**2 * (a**2 * n**2 * R33 - 16 * b**2 * m**2 * R11 * S112) * (2 * np.cos(m * np.pi) + np.cos(2 * m * np.pi))) * np.sin((m * np.pi) / 2)**2 * np.sin((n * np.pi) / 2)**2)

    V13 = (-1 / (8 * a * b) * m * n**2 * np.pi**3 * (4 * m * np.pi * (R11 + R21) -
          2 * (2 * R11 + R21) * np.sin(2 * m * np.pi) + R11 * np.sin(4 * m * np.pi)))

    V14 = (-1 / (8 * a**3 * b**3 * m) * (a**4 * n**4 * np.pi**3 * R31 * S114 +
          b**4 * (m**4 * np.pi**3 * R31 * S112 - a**3 * m * S16 + a * m**3 * np.pi**2 * S17) +
          a**2 * b**2 * m * n**2 * np.pi**2 * (m * np.pi * R31 * S113 + a * S18)) * (-2 * m * np.pi + np.sin(2 * m * np.pi)))

    V15 = (-1 / (8 * a**3 * b**3 * m) * (np.pi**2 * (b**4 * m**4 * np.pi * R32 * S112 +
          a**2 * b**2 * m**2 * n * (b * S111 + n * np.pi * R32 * S113) +
          a**4 * n**3 * (b * S110 + n * np.pi * R32 * S114)) - a**4 * b**3 * n * S19) * (-2 * m * np.pi + np.sin(2 * m * np.pi)))

    V16 = (-1 / (3 * a * b) * 4 * m * n * np.pi**2 * R31 * (1 + 2 * (-1)**n + 2 * np.cos(m * np.pi) + np.cos(2 * m * np.pi)) * np.sin((m * np.pi) / 2)**2 * np.sin((n * np.pi) / 2)**2)

    V17 = (-1 / (3 * a * b) * 4 * m * n * np.pi**2 * R32 * (1 + 2 * (-1)**n + 2 * np.cos(m * np.pi) + np.cos(2 * m * np.pi)) * np.sin((m * np.pi) / 2)**2 * np.sin((n * np.pi) / 2)**2)

    V18 = -(b * m * np.pi * (2 * m * np.pi - np.sin(2 * m * np.pi))) / (8 * a)
    V19 = -(a * n**2 * np.pi * (2 * m * np.pi - np.sin(2 * m * np.pi))) / (8 * b * m)

    V21 = (-1 / (8 * a**2 * b) * (b**2 * m**2 * np.pi**2 * (S22 + R33 * S28) +
          a**2 * (-b**2 * S23 + n**2 * np.pi**2 * (S21 + R33 * S29))) * (2 * m * np.pi + np.sin(2 * m * np.pi)))

    V22 = (16 * (-1 + (-1)**n) * b * m**2 * np.pi * R11 * S28 * (-1 + np.cos(m * np.pi)**3)) / (3 * a**2 * n)

    V23 = (-1 / (8 * a**2 * b * m * np.pi) * (a * b**2 * m**2 * np.pi**2 * S24 +
          a**3 * (n**2 * np.pi**2 * S25 - b**2 * S26) + b**2 * m**3 * np.pi**3 * R31 * S28 +
          a**2 * m * n**2 * np.pi**3 * R31 * S29) * (2 * m * np.pi + np.sin(2 * m * np.pi)))

    V24 = (-1 / (8 * a**2 * b) * np.pi * (b**2 * m**2 * np.pi * R32 * S28 +
          a**2 * n * (b * S27 + n * np.pi * R32 * S29)) * (2 * m * np.pi + np.sin(2 * m * np.pi)))

    V31 = (1 / (8 * a * b**2 * m) * n * (a**2 * n**2 * np.pi**2 * (S32 + R33 * S38) +
          b**2 * (-a**2 * S33 + m**2 * np.pi**2 * (S31 + R33 * S39))) * (-2 * m * np.pi + np.sin(2 * m * np.pi)))

    V32 = (16 * (-1 + (-1)**n) * a * n**2 * np.pi * R21 * S38 * (-1 + np.cos(m * np.pi))) / (3 * b**2 * m)

    V33 = (-1 / (8 * a * b**2 * m) * n * np.pi * (a * b**2 * m * S34 +
          a**2 * n**2 * np.pi * R31 * S38 + b**2 * m**2 * np.pi * R31 * S39) * (2 * m * np.pi - np.sin(2 * m * np.pi)))

    V34 = (1 / (8 * a * b**2 * m * np.pi) * (a**2 * b * n**2 * np.pi**2 * S36 +
          b**3 * (m**2 * np.pi**2 * S35 - a**2 * S37) + a**2 * n**3 * np.pi**3 * R32 * S38 +
          b**2 * m**2 * n * np.pi**3 * R32 * S39) * (-2 * m * np.pi + np.sin(2 * m * np.pi)))

    ###### %% Eq. (34)
    P11 = -((V24 * V31 - V21 * V34) / (V24 * V33 - V23 * V34))
    P12 = -((V24 * V32 - V22 * V34) / (V24 * V33 - V23 * V34))
    P21 = -((-V23 * V31 + V21 * V33) / (V24 * V33 - V23 * V34))
    P22 = -((-V23 * V32 + V22 * V33) / (V24 * V33 - V23 * V34))

    ###### %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    lamda0 = (-1) * (S2 + V11 + P11 * V14 + P21 * V15) * (Chi1 * V18 + Chi2 * V19)**(-1)
    lamda1 = (-1) * h * (V12 + P12 * V14 + P22 * V15 + P11 * V16 + P21 * V17) * (Chi1 * V18 + Chi2 * V19)**(-1)
    lamda2 = (-1) * h**2 * (V13 + P12 * V16 + P22 * V17) * (Chi1 * V18 + Chi2 * V19)**(-1)


    # ============================================================================
    # Normalized Results (Eqs. 38-39) 
    # ============================================================================
    ### lamda = Lambda0*Wn
    Lambda0 = lamda0 * a**2 / (ECu * h**3)
    Lambda1 = lamda1 * a**2 / (ECu * h**3)
    Lambda2 = lamda2 * a**2 / (ECu * h**3)

    # ---------------------------------------------------------------------------

    return Lambda0, Lambda1, Lambda2


In [3]:
import numpy as np
import time

# ====== bạn tự chỉnh các mảng này tuỳ ý ======
var1_grid  = np.array([10,15,20,25])              # a/h
var2_grid  = np.array([1.0, 1.5, 2 , 2.5])           # b/a
var3_grid  = np.array([300,400,500,600])           # T (K)
var4_grid  = np.array([25,50,75,100])             # degree Gori folding / H atomics
var5_grid  = np.array([0.5,1,1.5,2])           # wt% GORi
var6_grid  = np.array([1, 2, 3, 4])              # DT: 1=U,2=X,3=O,4=A
var7_grid  = np.array([5,10,15,20])                # Kun
var8_grid  = np.array([5,10,15,20])               # Ksn
var9_grid  = np.array([5,10,15,20])               # Kln
var10_grid = np.array([1, 2])                 # CL: 1,2
# ============================================

def generate_data_from_grids(
    var1_grid, var2_grid, var3_grid, var4_grid, var5_grid,
    var6_grid, var7_grid, var8_grid, var9_grid, var10_grid,
    num_points=50
):
    # Tạo lưới 10 chiều
    grids = np.meshgrid(
        var1_grid, var2_grid, var3_grid, var4_grid, var5_grid,
        var6_grid, var7_grid, var8_grid, var9_grid, var10_grid,
        indexing='ij'
    )
    # Ravel từng grid và xếp chồng thành (N,10)
    X = np.stack([g.ravel() for g in grids], axis=1)
    num_samples = X.shape[0]

    print(f"num_samples (tự sinh từ grid) = {num_samples}")

    # Chuẩn bị 2 head
    y_headA = np.zeros(num_samples)               # critical load = Lambda0
    y_headB = np.zeros((num_samples, num_points)) # 50 điểm parabol
    Wn = np.linspace(0.0, 1.0, num_points)

    # Đo thời gian tổng
    t0 = time.perf_counter()

    # Bao lâu thì in 1 lần (ở đây ~5% số mẫu)
    log_every = 200

    for i in range(num_samples):
        v1, v2, v3, v4, v5, v6, v7, v8, v9, v10 = X[i]

        # Hàm này là block analytical của bạn
        Lambda0, Lambda1, Lambda2 = compute_lambda_coeffs(
            v1, v2, v3, v4, v5, v6, v7, v8, v9, v10
        )

        # Head A: chỉ lấy critical load
        y_headA[i] = Lambda0

        # Head B: 50 điểm từ parabol
        y_headB[i, :] = Lambda0 + Lambda1 * Wn + Lambda2 * (Wn**2)

        # --------- In tiến trình + configs/s ---------
        if (i + 1) % log_every == 0 or (i + 1) == num_samples:
            t_now = time.perf_counter()
            elapsed_s = t_now - t0
            configs_done = i + 1
            percent = 100.0 * configs_done / num_samples
            speed = configs_done / elapsed_s if elapsed_s > 0 else float('inf')

            print(
                f"[{configs_done}/{num_samples}] "
                f"{percent:6.2f}%  |  {speed:8.2f} configs/s"
            )

    t1 = time.perf_counter()
    total_s = t1 - t0
    total_h = total_s / 3600.0

    print(
        f"Generate xong {num_samples} sample trong {total_s:.3f} s "
        f"(~{total_h:.6f} h)"
    )

    return X, y_headA, y_headB, Wn

X, yA, yB, Wn = generate_data_from_grids(
    var1_grid, var2_grid, var3_grid, var4_grid, var5_grid,
    var6_grid, var7_grid, var8_grid, var9_grid, var10_grid,
    num_points=50
)   

num_samples (tự sinh từ grid) = 524288
[200/524288]   0.04%  |    275.27 configs/s
[400/524288]   0.08%  |    279.95 configs/s
[600/524288]   0.11%  |    280.45 configs/s
[800/524288]   0.15%  |    279.26 configs/s
[1000/524288]   0.19%  |    277.66 configs/s
[1200/524288]   0.23%  |    275.88 configs/s
[1400/524288]   0.27%  |    274.46 configs/s
[1600/524288]   0.31%  |    275.04 configs/s
[1800/524288]   0.34%  |    275.49 configs/s
[2000/524288]   0.38%  |    274.76 configs/s
[2200/524288]   0.42%  |    274.52 configs/s
[2400/524288]   0.46%  |    272.61 configs/s
[2600/524288]   0.50%  |    273.44 configs/s
[2800/524288]   0.53%  |    274.27 configs/s
[3000/524288]   0.57%  |    274.88 configs/s
[3200/524288]   0.61%  |    274.81 configs/s
[3400/524288]   0.65%  |    275.42 configs/s
[3600/524288]   0.69%  |    276.10 configs/s
[3800/524288]   0.72%  |    276.64 configs/s
[4000/524288]   0.76%  |    276.75 configs/s
[4200/524288]   0.80%  |    276.41 configs/s
[4400/524288]   0.84

In [4]:
import pandas as pd
import numpy as np

def save_combined_dataset(X, y_headA, y_headB, Wn, prefix="fg_goeam_10var"):
    """
    Ghi file CSV:
      - Input (var1..var10): Giữ nguyên giá trị gốc (không ép format).
      - Output (Fcr, lam1..50): Bắt buộc chính xác 6 chữ số thập phân (%.6f).
    """
    
    X = np.asarray(X)
    y_headA = np.asarray(y_headA).reshape(-1, 1)
    y_headB = np.asarray(y_headB)
    Wn = np.asarray(Wn)

    num_samples, num_vars = X.shape
    _, num_points = y_headB.shape

    # --- 1. Tạo DataFrame tổng ---
    col_vars = [f"var{i}" for i in range(1, 11)]
    col_headA = ["Fcr"]
    col_headB = [f"lam{i}" for i in range(1, num_points + 1)]

    # Ghép dữ liệu thô trước
    data_all = np.column_stack([X, y_headA, y_headB])
    all_columns = col_vars + col_headA + col_headB
    df = pd.DataFrame(data_all, columns=all_columns)

    # --- 2. Format riêng cho các cột Output (Head A & Head B) ---
    # Chỉ ép các cột này về dạng chuỗi "x.xxxxxx"
    cols_to_format = col_headA + col_headB 
    
    for col in cols_to_format:
        # map('{:.6f}'.format) sẽ đảm bảo luôn hiện 
        df[col] = df[col].map('{:.8f}'.format)

    # >>> Thêm đoạn xem trước 20 dòng đầu
    print("📊 Xem trước 20 dòng đầu của DataFrame:")
    print(df.head(20).to_string(index=False))
    print("-" * 80)
    
    # --- 3. Ghi file ---
    # Không dùng float_format global nữa để tránh ảnh hưởng Input
    file_csv = f"{prefix}MTH.csv"
    df.to_csv(file_csv, index=False)
    
    print(f" Đã ghi file tổng: {file_csv}")
    print(f"   (Input giữ nguyên, Output chuẩn 6 số thập phân)")

    # --- 4. Ghi file Wn (Meta) ---
    df_Wn = pd.DataFrame({
        "col_name": col_headB,
        "Wn_value": Wn
    })
    # 
    df_Wn.to_csv(f"{prefix}_Wn.csv", index=False, float_format='%.8f')

In [5]:
save_combined_dataset(X, yA, yB, Wn, prefix="Dataset")


📊 Xem trước 20 dòng đầu của DataFrame:
 var1  var2  var3  var4  var5  var6  var7  var8  var9  var10        Fcr       lam1       lam2       lam3       lam4       lam5       lam6       lam7       lam8       lam9      lam10      lam11      lam12      lam13      lam14      lam15      lam16      lam17      lam18      lam19      lam20      lam21      lam22      lam23      lam24      lam25      lam26      lam27      lam28      lam29      lam30      lam31      lam32      lam33      lam34      lam35      lam36      lam37      lam38      lam39      lam40      lam41      lam42      lam43      lam44      lam45      lam46      lam47      lam48      lam49      lam50
 10.0   1.0 300.0  25.0   0.5   1.0   5.0   5.0   5.0    1.0 4.78428181 4.78428181 4.78488077 4.78667766 4.78967247 4.79386521 4.79925588 4.80584447 4.81363098 4.82261542 4.83279779 4.84417808 4.85675629 4.87053243 4.88550650 4.90167849 4.91904841 4.93761625 4.95738202 4.97834571 5.00050733 5.02386687 5.04842434 5.07417973 5.10113305 5.1